# 44 - Feature Engineering (Sale y Rent)

Este notebook crea features sobre copias de `sale_homes_clean_outliers.csv` y `rent_homes_clean_outliers.csv`, y exporta resultados a `data/gold/final_sale.csv` y `data/gold/final_rent.csv`.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed" / "idealistaAPI").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/processed/idealistaAPI")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "idealistaAPI"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

SALE_IN = DATA_DIR / "sale_homes_clean_outliers.csv"
RENT_IN = DATA_DIR / "rent_homes_clean_outliers.csv"

SALE_OUT = GOLD_DIR / "final_sale.csv"
RENT_OUT = GOLD_DIR / "final_rent.csv"

sale_df = pd.read_csv(SALE_IN)
rent_df = pd.read_csv(RENT_IN)

print(f"SALE input: {SALE_IN} -> {sale_df.shape}")
print(f"RENT input: {RENT_IN} -> {rent_df.shape}")

In [ ]:
UNIFAMILIAR_VALUES = {
    "chalet",
    "countryhouse",
    "singlefamily",
    "house",
    "villa",
    "townhouse",
    "detachedhouse",
    "adosado",
}

DISTANCE_COLS = [
    "distancia_min_playa_km",
    "distancia_min_supermercado_km",
    "distancia_min_colegio_km",
]

def normalize_boolean(series: pd.Series) -> pd.Series:
    true_values = {"true", "t", "1", "yes", "y", "si", "s"}
    false_values = {"false", "f", "0", "no", "n"}

    def _convert(value):
        if pd.isna(value):
            return 0
        if isinstance(value, (bool, np.bool_)):
            return int(value)
        if isinstance(value, (int, np.integer, float, np.floating)):
            if pd.isna(value):
                return 0
            return 1 if value >= 1 else 0

        text = str(value).strip().lower()
        if text in true_values:
            return 1
        if text in false_values:
            return 0
        return 0

    return series.map(_convert).astype(int)

def parse_planta(value) -> int:
    if pd.isna(value):
        return 0
    text = str(value).strip().lower()

    if text in {"bj", "bajo", "entresuelo", "sotano", "ss", "st"}:
        return 0

    match = re.search(r"-?\d+", text)
    if match:
        return int(match.group())

    return 0

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return r * c

def engineer_features(df_input: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df_input.copy()

    # Conversiones numericas base
    numeric_cols = [
        "precio",
        "superficie_construida_m2",
        "numero_dormitorios",
        "numero_banos",
        "latitud",
        "longitud",
        "planta",
        *DISTANCE_COLS,
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Solo casos con precio y superficie positivos para logs y ratios
    df = df[df["precio"] > 0].copy()
    df = df[df["superficie_construida_m2"] > 0].copy()

    # Imputacion simple para numericas necesarias
    for col in ["numero_dormitorios", "numero_banos", "latitud", "longitud", *DISTANCE_COLS]:
        if col in df.columns:
            med = df[col].median()
            if pd.isna(med):
                med = 0.0
            df[col] = df[col].fillna(med)

    # Booleanas a 0/1
    for col in ["es_exterior", "tiene_ascensor", "tiene_garaje", "obra_nueva"]:
        if col in df.columns:
            df[col] = normalize_boolean(df[col])

    # Tipologia unificada: piso / unifamiliar
    tipologia_text = df.get("tipologia", pd.Series(index=df.index, dtype="string")).astype("string").str.lower().str.strip()
    df["tipologia_unificada"] = np.where(tipologia_text.isin(UNIFAMILIAR_VALUES), "unifamiliar", "piso")

    # Logs
    df["log_precio"] = np.log(df["precio"])
    df["log_superficie_construida_m2"] = np.log(df["superficie_construida_m2"])

    # Precio por m2 y media por municipio
    df["precio_m2"] = df["precio"] / df["superficie_construida_m2"]
    df["precio_m2_municipio_media"] = df.groupby("municipio")["precio_m2"].transform("mean")

    # Ratios y agregados de habitaciones
    df["ratio_dormitorios_superficie"] = df["numero_dormitorios"] / df["superficie_construida_m2"]
    df["ratio_banos_superficie"] = df["numero_banos"] / df["superficie_construida_m2"]

    df["habitaciones"] = df["numero_dormitorios"] + df["numero_banos"]
    df["ratio_habitaciones_superficie"] = df["habitaciones"] / df["superficie_construida_m2"]

    # Interacciones
    df["interaccion_superficie_banos"] = df["superficie_construida_m2"] * df["numero_banos"]
    df["interaccion_superficie_habitaciones"] = df["superficie_construida_m2"] * df["habitaciones"]

    # Interaccion planta * (1 - ascensor) para pisos
    if "planta" in df.columns:
        df["planta_num"] = df["planta"].apply(parse_planta)
    else:
        df["planta_num"] = 0

    es_piso = (df["tipologia_unificada"] == "piso").astype(int)
    df["interaccion_planta_sin_ascensor_piso"] = df["planta_num"] * (1 - df.get("tiene_ascensor", 0)) * es_piso

    # Cuadrados geograficos e interaccion
    df["latitud_2"] = df["latitud"] ** 2
    df["longitud_2"] = df["longitud"] ** 2
    df["interaccion_latitud_longitud"] = df["latitud"] * df["longitud"]

    # Centro del municipio + distancia haversine
    centro_lat = df.groupby("municipio")["latitud"].transform("mean")
    centro_lon = df.groupby("municipio")["longitud"].transform("mean")

    df["distancia_centro_municipio_km"] = haversine_km(
        df["latitud"],
        df["longitud"],
        centro_lat,
        centro_lon,
    )

    # Variables inversas de distancia + score de cercania
    for col in DISTANCE_COLS:
        df[f"inv_{col}"] = 1 / (1 + df[col])

    df["score_cercania_servicios"] = df[[f"inv_{c}" for c in DISTANCE_COLS]].mean(axis=1)

    # Cuadrados solicitados
    df["superficie_construida_m2_2"] = df["superficie_construida_m2"] ** 2
    df["numero_banos_2"] = df["numero_banos"] ** 2
    df["numero_dormitorios_2"] = df["numero_dormitorios"] ** 2
    df["habitaciones_2"] = df["habitaciones"] ** 2

    # One-hot encoding de categoricas
    cat_cols = ["tipologia_unificada", "subtipologia", "provincia", "municipio", "distrito"]
    for col in cat_cols:
        if col not in df.columns:
            df[col] = "desconocido"
        df[col] = df[col].fillna("desconocido").astype(str).str.strip()
        df.loc[df[col] == "", col] = "desconocido"

    df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

    # Asegurar que booleans queden en 0/1
    bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(int)

    print(f"{name}: shape final {df.shape}")
    return df


In [ ]:
sale_final = engineer_features(sale_df, "SALE")
rent_final = engineer_features(rent_df, "RENT")

sale_final.to_csv(SALE_OUT, index=False)
rent_final.to_csv(RENT_OUT, index=False)

print(f"Exportado: {SALE_OUT}")
print(f"Exportado: {RENT_OUT}")

In [ ]:
required_cols = [
    "log_precio",
    "log_superficie_construida_m2",
    "precio_m2",
    "precio_m2_municipio_media",
    "ratio_dormitorios_superficie",
    "ratio_banos_superficie",
    "habitaciones",
    "ratio_habitaciones_superficie",
    "interaccion_superficie_banos",
    "interaccion_superficie_habitaciones",
    "interaccion_planta_sin_ascensor_piso",
    "latitud_2",
    "longitud_2",
    "interaccion_latitud_longitud",
    "distancia_centro_municipio_km",
    "inv_distancia_min_playa_km",
    "inv_distancia_min_supermercado_km",
    "inv_distancia_min_colegio_km",
    "score_cercania_servicios",
    "superficie_construida_m2_2",
    "numero_banos_2",
    "numero_dormitorios_2",
    "habitaciones_2",
]

missing_sale = [c for c in required_cols if c not in sale_final.columns]
missing_rent = [c for c in required_cols if c not in rent_final.columns]

print(f"Columnas nuevas faltantes en SALE: {missing_sale}")
print(f"Columnas nuevas faltantes en RENT: {missing_rent}")

ohe_tipologia_sale = [c for c in sale_final.columns if c.startswith("tipologia_unificada_")]
ohe_tipologia_rent = [c for c in rent_final.columns if c.startswith("tipologia_unificada_")]

print(f"Dummies tipologia SALE: {ohe_tipologia_sale}")
print(f"Dummies tipologia RENT: {ohe_tipologia_rent}")

print("\nPreview SALE (columnas clave):")
preview_cols = ["precio", "superficie_construida_m2", "log_precio", "log_superficie_construida_m2", "precio_m2", "precio_m2_municipio_media", "score_cercania_servicios"]
print(sale_final[preview_cols].head(3).to_string(index=False))